# Advanced 03 — Cross-Domain Identity Federation & Interoperability for Agents

Scenario: a corporate claims agent delegates research to a partner agent operating in a foreign identity/workload trust domain.

In [ ]:
from dataclasses import dataclass
from datetime import datetime,timedelta,timezone
import copy, json, hashlib
import pandas as pd
import networkx as nx
NOW=datetime.now(timezone.utc)


## 1 — Domain-qualified identity

In [ ]:
corp=("https://id.corp.example","agent:claims")
partner=("https://id.partner.example","agent:research")
assert corp != partner


## 2 — Namespace collision

In [ ]:
a=("partner-a.example","agent:research")
b=("partner-b.example","agent:research")
print(a,b,a==b)


## 3 — Trust registry

In [ ]:
registry={
"partner.example":{"active":True,"issuer":"https://id.partner.example",
 "audiences":{"claims-mcp"},"agent_types":{"research"},"max_ttl":600},
"evil.example":{"active":False,"issuer":"https://id.evil.example",
 "audiences":set(),"agent_types":set(),"max_ttl":0}}
registry


## 4 — Foreign principal

In [ ]:
@dataclass(frozen=True)
class FederatedPrincipal:
    issuer:str
    subject:str
    trust_domain:str
    principal_type:str="agent"
p=FederatedPrincipal("https://id.partner.example","agent:research","partner.example")


## 5 — Validate federation relationship

In [ ]:
def federation_ok(p,audience,agent_type):
    pol=registry.get(p.trust_domain)
    if not pol or not pol["active"]: return False,"UNTRUSTED_DOMAIN"
    if p.issuer!=pol["issuer"]: return False,"ISSUER"
    if audience not in pol["audiences"]: return False,"AUDIENCE"
    if agent_type not in pol["agent_types"]: return False,"TYPE"
    return True,"VALID"
federation_ok(p,"claims-mcp","research")


## 6 — Directed trust graph

In [ ]:
g=nx.DiGraph()
g.add_edge("corp.example","partner.example",relation="accepts_identity_from")
g.add_edge("partner.example","vendor.example",relation="accepts_identity_from")
list(g.edges(data=True))


## 7 — Trust is not automatically transitive

In [ ]:
assert not g.has_edge("corp.example","vendor.example")
print("A→B and B→C does not imply A→C")


## 8 — Simplified OpenID Federation trust chain

In [ ]:
chain=[
 {"subject":"agent.partner.example","issuer":"partner.example","signature_valid":True,"time_valid":True},
 {"subject":"partner.example","issuer":"federation.example","signature_valid":True,"time_valid":True},
 {"subject":"federation.example","issuer":"federation.example","signature_valid":True,"time_valid":True}
]
accepted_anchor="federation.example"


## 9 — Verify chain to local anchor

In [ ]:
def verify_chain(chain,anchor):
    if chain[-1]["subject"]!=anchor:return False
    if not all(x["signature_valid"] and x["time_valid"] for x in chain):return False
    for child,parent in zip(chain,chain[1:]):
        if child["issuer"]!=parent["subject"]:return False
    return True
verify_chain(chain,accepted_anchor)


## 10 — Trust-chain substitution attack

In [ ]:
sub=copy.deepcopy(chain);sub[-1]["subject"]="attacker-anchor.example"
verify_chain(sub,accepted_anchor)


## 11 — Metadata policy

In [ ]:
metadata={"alg":"ES256","grant_types":{"authorization_code"},"token_endpoint_auth":"private_key_jwt"}
policy={"allowed_algs":{"ES256","EdDSA"},"required_grant":"authorization_code"}
assert metadata["alg"] in policy["allowed_algs"] and policy["required_grant"] in metadata["grant_types"]


## 12 — Trust marks are inputs, not permissions

In [ ]:
trust_marks={"security-baseline-v2":{"issuer":"federation.example","valid":True}}
local_rule = trust_marks["security-baseline-v2"]["valid"]
print("Evidence accepted:",local_rule,"— still requires local authorization.")


## 13 — SPIFFE identities

In [ ]:
local_spiffe="spiffe://corp.example/prod/claims-agent"
foreign_spiffe="spiffe://partner.example/prod/research-agent"
assert local_spiffe != foreign_spiffe


## 14 — Preserve trust-domain ↔ bundle binding

In [ ]:
bundles={"corp.example":"corp-bundle-hash","partner.example":"partner-bundle-hash"}
def bundle_for(spiffe_id):
    domain=spiffe_id.split("/")[2]
    return bundles.get(domain)
bundle_for(foreign_spiffe)


## 15 — Bundle substitution

In [ ]:
attacked=copy.deepcopy(bundles)
attacked["partner.example"]="attacker-bundle"
assert attacked["partner.example"] != bundles["partner.example"]
print("Federation config integrity is security critical.")


## 16 — Directional SPIFFE federation

In [ ]:
federation_edges={("corp.example","partner.example")}
print("corp validates partner:",("corp.example","partner.example") in federation_edges)
print("partner validates corp:",("partner.example","corp.example") in federation_edges)


## 17 — Key rotation

In [ ]:
bundle={"domain":"partner.example","keys":{"old","new"}}
bundle["keys"].remove("old")
print(bundle)


## 18 — Local authorization after authentication

In [ ]:
foreign_verified=True
local_relationship=True
delegation_valid=True
risk="low"
allow=foreign_verified and local_relationship and delegation_valid and risk=="low"
allow


## 19 — Never map foreign roles directly

In [ ]:
foreign_claims={"role":"admin"}
local_roles=set()
assert "admin" not in local_roles


## 20 — Cross-domain delegation

In [ ]:
delegation={"issuer":"agent:claims@corp.example",
"subject":"agent:research@partner.example","audience":"claims-mcp",
"actions":{"knowledge.search"},"resources":{"claim:483"},
"expires":NOW+timedelta(minutes=10),"max_depth":0}


## 21 — Authority intersection

In [ ]:
foreign_identity_actions={"knowledge.search","claim.read"}
delegated=delegation["actions"]
federation_policy={"knowledge.search"}
local_resource_policy={"knowledge.search"}
effective=foreign_identity_actions & delegated & federation_policy & local_resource_policy
effective


## 22 — Third-party agent onboarding

In [ ]:
onboarding=pd.DataFrame([
["Provider","Partner Inc."],["Trust domain","partner.example"],
["Agent","agent:research"],["Purpose","claims research"],
["Risk tier","medium"],["Review","quarterly"]
],columns=["field","value"])
onboarding


## 23 — MCP Protected Resource Metadata

In [ ]:
prm={"resource":"https://mcp.corp.example",
"authorization_servers":["https://id.corp.example","https://partner-as.example"]}
prm


## 24 — Authorization-server selection policy

In [ ]:
approved_as={"https://id.corp.example"}
selected=[x for x in prm["authorization_servers"] if x in approved_as]
selected


## 25 — Resource binding

In [ ]:
token={"iss":"https://id.corp.example","aud":"https://mcp.corp.example"}
assert token["aud"]==prm["resource"]


## 26 — Identity translation gateway

In [ ]:
translated={"local_subject":"federated:partner.example:agent:research",
"original_issuer":p.issuer,"original_subject":p.subject,
"aud":"claims-mcp","ttl_seconds":300,"scope":["knowledge.search"]}
translated


## 27 — Malicious issuer attack

In [ ]:
evil=FederatedPrincipal("https://id.evil.example","agent:research","evil.example")
federation_ok(evil,"claims-mcp","research")


## 28 — Stale metadata

In [ ]:
cache={"fetched_at":NOW,"ttl":timedelta(minutes=5)}
def fresh(c,now):return now < c["fetched_at"]+c["ttl"]
fresh(cache,NOW+timedelta(minutes=2)),fresh(cache,NOW+timedelta(minutes=10))


## 29 — Federation termination

In [ ]:
offboard=copy.deepcopy(registry)
offboard["partner.example"]["active"]=False
print(offboard["partner.example"]["active"])


## 30 — Evidence

In [ ]:
evidence={"foreign_issuer":p.issuer,"foreign_subject":p.subject,
"trust_domain":p.trust_domain,"trust_anchor":accepted_anchor,
"audience":"claims-mcp","delegation_subject":delegation["subject"],
"local_decision":"allow","timestamp":NOW.isoformat()}
evidence


## 31 — Adversarial test matrix

Test:

- valid signature from unknown issuer;
- namespace collision;
- trust-anchor substitution;
- bundle substitution;
- metadata algorithm downgrade;
- stale JWKS/bundle;
- cross-domain role injection;
- over-broad delegation;
- wrong audience/resource;
- malicious authorization server in MCP metadata;
- token passthrough;
- terminated federation relationship;
- unintended trust transitivity.


# Capstone

Implement:

```text
Alice @ Corp
   ↓
Claims Agent @ corp.example
   ↓ bounded delegation
Research Agent @ partner.example
   ↓ OAuth
Corp MCP Knowledge Server
```

Requirements:

1. partner identity is domain-qualified;
2. issuer/trust domain is locally approved;
3. OpenID/SPIFFE trust terminates at configured trust;
4. foreign roles never become local roles automatically;
5. delegation is audience/resource/action/time bounded;
6. effective authority is an intersection;
7. MCP authorization-server selection is policy-controlled;
8. tokens are resource-bound;
9. federation state has freshness rules;
10. partner offboarding invalidates trust;
11. evidence records foreign and local identity;
12. negative tests cover trust substitution and namespace attacks.


# Review questions

1. What is the difference between federation and universal trust?
2. Why is authentication federation separate from authorization?
3. What is an OpenID Federation trust anchor?
4. What is an Entity Statement?
5. What does metadata policy do?
6. Why is a trust mark not a permission?
7. What is a SPIFFE trust domain?
8. Why must the trust-domain/bundle binding be preserved?
9. Why is SPIFFE federation directional?
10. Why is trust not automatically transitive?
11. Why must foreign identity namespaces be preserved?
12. Why should foreign roles not map directly to local roles?
13. What should happen when federation metadata is stale?
14. How should partner offboarding affect cached decisions?
15. How does MCP Protected Resource Metadata affect cross-domain authorization?
16. What evidence is required to explain why a foreign agent was trusted?
